# Homework 2 — Problem 1

Karl Rummler

## part c

1. libraries and random seed

In [ ]:
from pathlib import Path
import copy

import numpy as np
import pandas as pd
import torch
from torch import nn
from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, brier_score_loss, log_loss
from sklearn.metrics import accuracy_score
from IPython.display import display


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


2. data and train validation test split

In [ ]:
data_dir = Path("data") if Path("data").is_dir() else Path(".")
if not (data_dir / "taxi_trips.parquet").exists():
    from google.colab import files
    files.upload()
    data_dir = Path(".")

trips = pd.read_parquet(data_dir / "taxi_trips.parquet")
feature_names = trips.columns.drop("trip_duration").tolist()

# Look at the data
display(trips.head())
display(trips["trip_duration"].describe(percentiles=[0.5, 0.9, 0.99]).to_frame())


# Create training / validation / test split
X = trips[feature_names].to_numpy(dtype=np.float32)
y = trips["trip_duration"].to_numpy(dtype=np.float32)
X_train, X_other, y_train, y_other = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_other, y_other, test_size=0.50, random_state=SEED
)
print("Train / validation / test:", len(y_train), len(y_val), len(y_test))


,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration,min_of_day,day_of_week,day_of_year
187173,0.270744,1.425051,0.466831,0.776192,7.965142,-0.942057,-1.047619,1.180855
179655,0.085681,0.306079,-0.226274,0.124111,10.680277,0.778703,-1.047619,-0.311290
120048,-0.555533,-0.934806,-0.022489,-0.548561,5.716184,-1.183016,-1.047619,1.316504
62587,0.069718,0.035804,0.258900,0.317284,4.266709,-0.908008,0.481657,0.018144
212063,-0.234482,0.074616,0.115836,0.258310,11.629685,1.066806,-0.537860,0.521985


,trip_duration
count,523554.000000
mean,10.413046
std,7.753304
min,0.033386
50%,8.631456
90%,19.840349
99%,37.124424
max,227.942612


Train / validation / test: 366487 78533 78534


3. baseline models and label scaling

In [ ]:
# Ridge regression for our baseline model

train_mean = float(y_train.mean())
train_sd = float(y_train.std())
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
ridge_sd = float(np.sqrt(np.mean((y_train - ridge.predict(X_train)) ** 2)))


# Gaus [N]egative [L]og [L]ikelihood loss fn
def gaussian_nll(y_true, mean, sd):
    return float(np.mean(
        0.5 * np.log(2 * np.pi) + np.log(sd)
        + 0.5 * ((y_true - mean) / sd) ** 2
    ))

baseline_val = pd.DataFrame([
    {"Model": "Constant Gaussian", "Validation NLL": gaussian_nll(y_val, train_mean, train_sd)},
    {"Model": "Ridge Gaussian", "Validation NLL": gaussian_nll(y_val, ridge.predict(X_val), ridge_sd)},
])
display(baseline_val.round(4))

y_train_scaled = (y_train - train_mean) / train_sd
y_val_scaled = (y_val - train_mean) / train_sd


,Model,Validation NLL
0,Constant Gaussian,3.4661
1,Ridge Gaussian,3.4369


4. neural network predicting a mean and standard deviation

In [ ]:
class GaussianMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # input_dim -> 64 (relu) -> 64 (relu) -> 2 [mean, sd = softplus(raw_sd) + 1e-3]
        self.hidden = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
        )
        self.output = nn.Linear(64, 2)
        self.softplus = nn.Softplus()

    def forward(self, x):
        raw = self.output(self.hidden(x))
        mean = raw[:, 0]
        sd = self.softplus(raw[:, 1]) + 1e-3
        return mean, sd


### 5. gaussian negative log likelihood loss

In [ ]:
def normal_loss(y_true, mean, sd):
    return (0.5 * np.log(2 * np.pi) + torch.log(sd)
            + 0.5 * ((y_true - mean) / sd).square()).mean()


### 6. model optimizer and training tensors

In [ ]:
model = GaussianMLP(X_train.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_x = torch.tensor(X_train)
train_y = torch.tensor(y_train_scaled)
val_x = torch.tensor(X_val)
val_y = torch.tensor(y_val_scaled)


### 7. training with validation and early stopping

In [ ]:
batch_size = 4096
max_epochs = 80
patience = 10
best_loss = float("inf")
best_state = None
stale_epochs = 0

for epoch in range(1, max_epochs + 1):
    model.train()
    order = torch.randperm(len(train_y))
    total_loss = 0.0

    for start in range(0, len(train_y), batch_size):
        batch = order[start:start + batch_size]
        optimizer.zero_grad()
        mean, sd = model(train_x[batch])
        loss = normal_loss(train_y[batch], mean, sd)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch)

    model.eval()
    with torch.no_grad():
        val_mean, val_sd = model(val_x)
        val_loss = normal_loss(val_y, val_mean, val_sd).item()

    train_loss = total_loss / len(train_y)
    if val_loss < best_loss:
        best_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        stale_epochs = 0
    else:
        stale_epochs += 1

    print(f"Epoch {epoch:2d}: train NLL={train_loss:.4f}, validation NLL={val_loss:.4f}")
    if stale_epochs >= patience:
        break

model.load_state_dict(best_state)
print("Selected epoch:", best_epoch)


Epoch  1: train NLL=1.2668, validation NLL=1.0500
Epoch  2: train NLL=0.9312, validation NLL=0.8251
Epoch  3: train NLL=0.7759, validation NLL=0.7617
Epoch  4: train NLL=0.7320, validation NLL=0.7237
Epoch  5: train NLL=0.7061, validation NLL=0.7043
Epoch  6: train NLL=0.6897, validation NLL=0.6881
Epoch  7: train NLL=0.6755, validation NLL=0.6751
Epoch  8: train NLL=0.6636, validation NLL=0.6672
Epoch  9: train NLL=0.6545, validation NLL=0.6662
Epoch 10: train NLL=0.6480, validation NLL=0.6508
Epoch 11: train NLL=0.6420, validation NLL=0.6523
Epoch 12: train NLL=0.6365, validation NLL=0.6520
Epoch 13: train NLL=0.6316, validation NLL=0.6414
Epoch 14: train NLL=0.6271, validation NLL=0.6358
Epoch 15: train NLL=0.6241, validation NLL=0.6311
Epoch 16: train NLL=0.6193, validation NLL=0.6348
Epoch 17: train NLL=0.6152, validation NLL=0.6288
Epoch 18: train NLL=0.6133, validation NLL=0.6255
Epoch 19: train NLL=0.6105, validation NLL=0.6213
Epoch 20: train NLL=0.6067, validation NLL=0.6207


### 8. predictions converted back to minutes

In [ ]:
def predict_distribution(X_new):
    model.eval()
    with torch.no_grad():
        mean_scaled, sd_scaled = model(torch.tensor(X_new, dtype=torch.float32))
    mean = train_mean + train_sd * mean_scaled.numpy()
    sd = train_sd * sd_scaled.numpy()
    return mean, sd

val_mean, val_sd = predict_distribution(X_val)
display(pd.concat([
    baseline_val,
    pd.DataFrame([{"Model": "Gaussian MLP", "Validation NLL": gaussian_nll(y_val, val_mean, val_sd)}])
], ignore_index=True).round(4))


,Model,Validation NLL
0,Constant Gaussian,3.4661
1,Ridge Gaussian,3.4369
2,Gaussian MLP,2.6344


## part e

### 9. test probabilities and prediction scores

In [ ]:
test_mean, test_sd = predict_distribution(X_test)
distributions = {
    "Constant Gaussian": (np.full(len(y_test), train_mean), np.full(len(y_test), train_sd)),
    "Ridge Gaussian": (ridge.predict(X_test), np.full(len(y_test), ridge_sd)),
    "Gaussian MLP": (test_mean, test_sd),
}
duration_rows = []
for name, (mean, sd) in distributions.items():
    duration_rows.append({
        "Model": name,
        "Test NLL": gaussian_nll(y_test, mean, sd),
        "Test RMSE (minutes)": np.sqrt(mean_squared_error(y_test, mean)),
    })
duration_results = pd.DataFrame(duration_rows).set_index("Model")
display(duration_results.round(4))

event = (y_test < 45).astype(int)
event_probabilities = {
    "Training event frequency": np.full(len(y_test), np.mean(y_train < 45))
}
for name, (mean, sd) in distributions.items():
    event_probabilities[name] = norm.cdf((45 - mean) / sd)

# Creating a pandas dffun for nice display
probability_rows = []
for name, probability in event_probabilities.items():
    prediction = (probability >= 0.5).astype(int)
    probability_rows.append({
        "Model": name,
        "Brier score": brier_score_loss(event, probability),
        "Log-loss": log_loss(event, np.clip(probability, 1e-7, 1 - 1e-7), labels=[0, 1]),
        "Accuracy": accuracy_score(event, prediction),
    })
probability_results = pd.DataFrame(probability_rows).set_index("Model")
display(probability_results.round(6))
print(f"Test trips under 45 minutes: {event.mean():.3%}")
print(f"Test trips at least 45 minutes: {(1 - event).sum():,}")


,Test NLL,Test RMSE (minutes)
Model,,
Constant Gaussian,3.4675,7.7569
Ridge Gaussian,3.4398,7.5445
Gaussian MLP,2.7562,4.0250


,Brier score,Log-loss,Accuracy
Model,,,
Training event frequency,0.004437,0.028572,0.995543
Constant Gaussian,0.004457,0.055276,0.995543
Ridge Gaussian,0.004455,0.045921,0.995543
Gaussian MLP,0.002941,0.018074,0.996587


Test trips under 45 minutes: 99.554%
Test trips at least 45 minutes: 350
